# Quickstart - Inference

This notebook covers a simple client usage, including the following points:
- List available models.
- Use the SambaNova inference adaptor to interact with cloud-based LLM chat models.
- Implement a chat loop conversation using the SambaNova inference adaptor.

Run inference via chat completions with the OGX Python SDK.

Please refer to the [OGX quickstart documentation](https://ogx-ai.github.io/docs/getting_started/quickstart) for further details.

In [1]:
# Imports
import os
import sys

from ogx_client import OgxClient


## Setup

In [2]:
# Create HTTP client
OGX_PORT = 8321
client = OgxClient(base_url=f"http://localhost:{OGX_PORT}")


In [3]:
# List available models
models = client.models.list()
print("--- Available models: ---")
for m in models.data:
    print(f"- {m.id}")
print()


In [ ]:
# Choose an inference model from the previous list
model = "sambanova/sambanova/Meta-Llama-3.3-70B-Instruct"

## Create a Chat Completion Request
Use the `chat_completion` function to define the conversation context. Each message you include should have a specific role and content:

In [5]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a friendly assistant."},
        {"role": "user", "content": "Write a two-sentence poem about llamas."},
    ],
)

print(response.choices[0].message.content)


## Conversation Loop
To create a continuous conversation loop, where users can input multiple messages in a session, use the following structure. This example runs an asynchronous loop, ending when the user types 'exit,' 'quit,' or 'bye.'

In [6]:
import asyncio
from ogx_client import OgxClient
from termcolor import cprint

async def chat_loop():
    while True:
        user_input = input("User> ")
        cprint(f"> User: {user_input}", "green")
        if user_input.lower() in ["exit", "quit", "bye"]:
            cprint("Ending conversation. Goodbye!", "yellow")
            break

        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": user_input}],
        )
        cprint(f"> Response: {response.choices[0].message.content}", "cyan")


# Run the chat loop in a Jupyter Notebook cell using await
await chat_loop()
# To run it in a python file, use this line instead
# asyncio.run(chat_loop())


## Conversation History
Maintaining a conversation history allows the model to retain context from previous interactions. Use a list to accumulate messages, enabling continuity throughout the chat session.

In [7]:
async def chat_loop():
    conversation_history = []
    while True:
        user_input = input("User> ")
        cprint(f"> User: {user_input}", "green")
        if user_input.lower() in ["exit", "quit", "bye"]:
            cprint("Ending conversation. Goodbye!", "yellow")
            break

        conversation_history.append({"role": "user", "content": user_input})

        response = client.chat.completions.create(
            model=model,
            messages=conversation_history,
        )
        assistant_content = response.choices[0].message.content
        cprint(f"> Response: {assistant_content}", "cyan")
        conversation_history.append({"role": "assistant", "content": assistant_content})


# Use `await` in the Jupyter Notebook cell to call the function
await chat_loop()
# To run it in a python file, use this line instead
# asyncio.run(chat_loop())


## Streaming Responses
OGX supports streaming via the OpenAI-compatible streaming API, returning partial
responses progressively as they are generated.


In [8]:
from termcolor import cprint

async def run_main():
    message = {"role": "user", "content": "Please write me a 3 sentence poem about llamas."}
    cprint(f'User> {message["content"]}', "green")

    cprint("> Response: ", "cyan", end="")
    for chunk in client.chat.completions.create(
        model=model,
        messages=[message],
        stream=True,
    ):
        content = chunk.choices[0].delta.content or ""
        cprint(content, "cyan", end="", flush=True)
    print()


# In a Jupyter Notebook cell, use `await` to call the function
await run_main()
# To run it in a python file, use this line instead
# asyncio.run(run_main())
